In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from jColor.Color import Color
from jColor.Image import Image

In [ ]:
SAMPLE = 'M1'
MAX_PIXELS = 100_000
SILHOUETTE_SAMPLE_SIZE = 5_000
RANDOM_STATE = 0

image_path = f'imgs/{SAMPLE}.jpg'
jade_image = Image(str(image_path))
normalized_rgb, white_reference, foreground_mask = jade_image.RGB_Normalization()
foreground = foreground_mask.astype(bool)
lab_pixels = Color(normalized_rgb).Lab[foreground]

rng = np.random.default_rng(RANDOM_STATE)
if len(lab_pixels) > MAX_PIXELS:
    selected_indices = rng.choice(len(lab_pixels), size=MAX_PIXELS, replace=False)
    elbow_data = lab_pixels[selected_indices]
else:
    elbow_data = lab_pixels.copy()

In [ ]:
K_VALUES = np.arange(1, 11)
inertias = []
silhouette_scores = []

silhouette_rng = np.random.default_rng(RANDOM_STATE)
silhouette_indices = silhouette_rng.choice(
    len(elbow_data), size=min(SILHOUETTE_SAMPLE_SIZE, len(elbow_data)), replace=False
)
silhouette_data = elbow_data[silhouette_indices]

for k in K_VALUES:
    model = KMeans(
        n_clusters=int(k),
        init='k-means++',
        n_init=10,
        random_state=RANDOM_STATE,
    )
    model.fit(elbow_data)
    inertias.append(model.inertia_)

    subset_labels = model.labels_[silhouette_indices]
    n_labels = len(np.unique(subset_labels))
    score = np.nan
    if 2 <= n_labels < len(silhouette_data):
        score = silhouette_score(silhouette_data, subset_labels, metric='euclidean')
    silhouette_scores.append(score)
    print(f'k={k:2d} | inertia={model.inertia_:,.2f} | silhouette={score:.4f}')

inertias = np.asarray(inertias)
silhouette_scores = np.asarray(silhouette_scores)

elbow_table = pd.DataFrame({
    'k': K_VALUES,
    'inertia': inertias,
    'silhouette': silhouette_scores,
})


In [ ]:
x_normalized = (K_VALUES - K_VALUES.min()) / (K_VALUES.max() - K_VALUES.min())
y_normalized = (inertias - inertias.min()) / (inertias.max() - inertias.min())

start = np.array([x_normalized[0], y_normalized[0]])
end = np.array([x_normalized[-1], y_normalized[-1]])
points = np.column_stack((x_normalized, y_normalized))
line_vector = end - start
offsets = points - start
cross_magnitudes = np.abs(line_vector[0] * offsets[:, 1] - line_vector[1] * offsets[:, 0])
distances = cross_magnitudes / np.linalg.norm(line_vector)
suggested_k = int(K_VALUES[np.argmax(distances)])

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

axes[0].plot(K_VALUES, inertias, marker='o', color='darkblue')
axes[0].axvline(suggested_k, color='red', linestyle='--', label=f'Suggested elbow: k={suggested_k}')
axes[0].scatter(suggested_k, inertias[suggested_k - 1], color='red', s=80, zorder=3)
axes[0].set_xticks(K_VALUES)
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title(f'Elbow method — {SAMPLE}')
axes[0].grid(alpha=0.25)
axes[0].legend()

valid_silhouette = np.isfinite(silhouette_scores)
axes[1].plot(
    K_VALUES[valid_silhouette], silhouette_scores[valid_silhouette],
    marker='o', color='slateblue'
)
if np.any(valid_silhouette):
    best_index = np.flatnonzero(valid_silhouette)[np.argmax(silhouette_scores[valid_silhouette])]
    best_silhouette_k = int(K_VALUES[best_index])
    axes[1].axvline(
        best_silhouette_k, color='red', linestyle='--',
        label=f'Highest silhouette: k={best_silhouette_k}'
    )
    axes[1].legend()
axes[1].set_xticks(K_VALUES[K_VALUES >= 2])
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Mean silhouette score')
axes[1].set_title(f'Silhouette — {SAMPLE} (n={len(silhouette_data):,})')
axes[1].grid(alpha=0.25)

plt.show()
print(f'Suggested value from geometric distance: k = {suggested_k}')